In [8]:
# ============================================================
# RANDOM FOREST CLASSIFIER
# FULL IMPLEMENTATION + HYPERPARAMETER TUNING
# + STABILITY ANALYSIS + COMPUTATION TIME
# ============================================================

import numpy as np
import pandas as pd
import time
import pickle
import warnings
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")


# ============================================================
# LOAD DATA
# ============================================================

def load_cleaned_data(filename='data_cleaned.pkl'):
    print("Loading cleaned data...")
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    return data


# ============================================================
# THRESHOLD OPTIMIZATION (Youden Index)
# ============================================================

def find_optimal_threshold(y_true, y_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    J = tpr - fpr
    best_idx = np.argmax(J)
    return thresholds[best_idx]


# ============================================================
# CROSS VALIDATION
# ============================================================

def run_cross_validation_rf(X, y, preprocessor, rf_params, n_splits=10):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        start_time = time.time()

        X_train_raw = X.iloc[train_idx]
        X_test_raw = X.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        X_train = preprocessor.fit_transform(X_train_raw)
        X_test = preprocessor.transform(X_test_raw)

        model = RandomForestClassifier(
            n_estimators=rf_params["n_estimators"],
            max_depth=rf_params["max_depth"],
            min_samples_split=rf_params["min_samples_split"],
            min_samples_leaf=rf_params["min_samples_leaf"],
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_proba = model.predict_proba(X_test)[:, 1]
        threshold = find_optimal_threshold(y_test, y_proba)
        y_pred = (y_proba >= threshold).astype(int)

        fold_results.append({
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "AUC": roc_auc_score(y_test, y_proba),
            "Threshold": threshold,
            "Training Time (s)": time.time() - start_time
        })

    return pd.DataFrame(fold_results)


# ============================================================
# GRID SEARCH
# ============================================================

def grid_search_rf(X, y, preprocessor, param_grid, n_splits=10):

    combinations = list(product(
        param_grid["n_estimators"],
        param_grid["max_depth"],
        param_grid["min_samples_split"],
        param_grid["min_samples_leaf"]
    ))

    all_results = []

    print("\nGRID SEARCH RANDOM FOREST STARTED")
    print("=" * 60)

    for i, (n_est, depth, split, leaf) in enumerate(combinations, 1):

        print(f"[{i}/{len(combinations)}] "
              f"Trees={n_est}, Depth={depth}")

        rf_params = {
            "n_estimators": int(n_est),
            "max_depth": None if depth is None else int(depth),
            "min_samples_split": int(split),
            "min_samples_leaf": int(leaf)
        }

        cv_df = run_cross_validation_rf(X, y, preprocessor, rf_params, n_splits)

        all_results.append({
            "n_estimators": rf_params["n_estimators"],
            "max_depth": rf_params["max_depth"],
            "min_samples_split": rf_params["min_samples_split"],
            "min_samples_leaf": rf_params["min_samples_leaf"],
            "Accuracy": cv_df["Accuracy"].mean(),
            "Precision": cv_df["Precision"].mean(),
            "Recall": cv_df["Recall"].mean(),
            "F1 Score": cv_df["F1 Score"].mean(),
            "AUC": cv_df["AUC"].mean()
        })

    results_df = pd.DataFrame(all_results)

    results_df = results_df.sort_values(
        by=["F1 Score", "Recall"],
        ascending=False
    ).reset_index(drop=True)

    return results_df


# ============================================================
# STABILITY CALCULATION
# ============================================================

def calculate_stability(cv_df):

    metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC"]

    summary = []

    for metric in metrics:
        mean_val = cv_df[metric].mean()
        std_val = cv_df[metric].std()

        if mean_val != 0:
            cv_percent = (std_val / mean_val) * 100
        else:
            cv_percent = 0

        if cv_percent < 5:
            status = "SANGAT STABIL"
        elif cv_percent < 10:
            status = "STABIL"
        else:
            status = "KURANG STABIL"

        summary.append({
            "Metric": metric,
            "Mean": mean_val,
            "Std Dev": std_val,
            "CV (%)": cv_percent,
            "Stability": status
        })

    summary_df = pd.DataFrame(summary)

    # Tambah waktu komputasi
    time_mean = cv_df["Training Time (s)"].mean()
    time_total = cv_df["Training Time (s)"].sum()

    return summary_df, time_mean, time_total


# ============================================================
# MAIN
# ============================================================

def main():

    print("=" * 70)
    print("RANDOM FOREST - HYPERPARAMETER TUNING + STABILITY")
    print("=" * 70)

    data_loaded = load_cleaned_data("data_cleaned.pkl")

    data_cleaned = data_loaded['data_cleaned']
    preprocessor = data_loaded['preprocessor']

    X = data_cleaned.drop(columns=["diagnosis_lanjutan"])
    y = data_cleaned["diagnosis_lanjutan"].values

    print(f"\nDataset Info:")
    print(f"Jumlah Fitur : {X.shape[1]}")
    print(f"Jumlah Data  : {X.shape[0]}")

    param_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }

    tuning_results = grid_search_rf(X, y, preprocessor, param_grid)
    tuning_results.to_csv("rf_grid_search_results.csv", index=False)

    best_config = tuning_results.iloc[0]

    max_depth_value = best_config["max_depth"]

    if pd.isna(max_depth_value):
        max_depth_value = None
    else:
        max_depth_value = int(max_depth_value)

    rf_params_best = {
        "n_estimators": int(best_config["n_estimators"]),
        "max_depth": max_depth_value,
        "min_samples_split": int(best_config["min_samples_split"]),
        "min_samples_leaf": int(best_config["min_samples_leaf"])
    }

    print("\nBEST CONFIGURATION:")
    print(rf_params_best)

    cv_best = run_cross_validation_rf(X, y, preprocessor, rf_params_best)

    print("\nDETAIL 10-FOLD CROSS VALIDATION")
    print(cv_best.to_string(index=False))

    summary_df, time_mean, time_total = calculate_stability(cv_best)

    print("\nRINGKASAN PERFORMA + STABILITAS")
    print(summary_df.to_string(index=False))

    print("\nWAKTU KOMPUTASI:")
    print(f"Rata-rata per fold : {time_mean:.4f} detik")
    print(f"Total waktu        : {time_total:.4f} detik")

    summary_df.to_csv("rf_stability_summary.csv", index=False)

    print("\n✅ SELESAI")
    print("File disimpan:")
    print("- rf_grid_search_results.csv")
    print("- rf_stability_summary.csv")


if __name__ == "__main__":
    main()

RANDOM FOREST - HYPERPARAMETER TUNING + STABILITY
Loading cleaned data...

Dataset Info:
Jumlah Fitur : 13
Jumlah Data  : 457

GRID SEARCH RANDOM FOREST STARTED
[1/36] Trees=100, Depth=None
[2/36] Trees=100, Depth=None
[3/36] Trees=100, Depth=None
[4/36] Trees=100, Depth=None
[5/36] Trees=100, Depth=10
[6/36] Trees=100, Depth=10
[7/36] Trees=100, Depth=10
[8/36] Trees=100, Depth=10
[9/36] Trees=100, Depth=20
[10/36] Trees=100, Depth=20
[11/36] Trees=100, Depth=20
[12/36] Trees=100, Depth=20
[13/36] Trees=200, Depth=None
[14/36] Trees=200, Depth=None
[15/36] Trees=200, Depth=None
[16/36] Trees=200, Depth=None
[17/36] Trees=200, Depth=10
[18/36] Trees=200, Depth=10
[19/36] Trees=200, Depth=10
[20/36] Trees=200, Depth=10
[21/36] Trees=200, Depth=20
[22/36] Trees=200, Depth=20
[23/36] Trees=200, Depth=20
[24/36] Trees=200, Depth=20
[25/36] Trees=300, Depth=None
[26/36] Trees=300, Depth=None
[27/36] Trees=300, Depth=None
[28/36] Trees=300, Depth=None
[29/36] Trees=300, Depth=10
[30/36] Tree